<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/09_technical_mart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import re

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"

DATASETS_DIR = "datasets"
# RECONCILIAÇÃO: Herdando os dados com categorias do Código 5 para analisar o ecossistema completo
INPUT_FILENAME = f"5_{CONSOLE_NAME}_games_with_genres.xlsx"     # Base com notas e gêneros
OUTPUT_FILENAME = f"{CONSOLE_NAME}_technical_performance.xlsx"  # O artefato de BI Técnico (Sem número)

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def clean_human_strings(text):
    """Splits strings with corporate notes/multiple companies and returns only the primary name."""
    if not text or pd.isna(text):
        return "Unknown"

    text = str(text)
    # Limpa distribuidores regionais e notas corporativas ex: "Enix (NA)" -> "Enix"
    parts = re.split(r'\(|/|•', text)
    clean_name = parts[0].strip()

    # Remove qualquer resíduo de colchetes de notas que tenha sobrevivido
    clean_name = re.sub(r'\[.*\]', '', clean_name).strip()
    return clean_name

def pipeline_bi_technical_analysis(input_path, output_path):
    """
    Cleans up structural overhead and builds a lightweight dataset focused on
    analyzing the correlation between scores, primary genres, and release years.
    """
    print(f"🔬 [CODE 09] Manufacturing Technical Analysis Artifact: {CONSOLE_NAME.upper()}")

    # 1. Pipeline gatekeeper check
    if not os.path.exists(input_path):
        print(f"❌ Critical Error: Input file '{input_path}' not found. Please run Notebook 05 first.")
        return None

    # 2. Load the pre-filtered category data
    df = pd.read_excel(input_path)
    print(f"📊 Integrating {len(df)} scored games into the technical evaluation template...")

    # 3. Apply Humanized Cleaning to Publishers and Developers
    df['Publisher'] = df['Publisher'].apply(clean_human_strings)
    df['Developer'] = df['Developer'].apply(clean_human_strings)

    # 4. Feature Selection and Custom Ordering (Enforcing your explicit schema)
    # Target Schema: title, publisher, developer, score, category, release
    df_bi_tech = pd.DataFrame({
        'Title': df['T1_Clean'],
        'Publisher': df['Publisher'],
        'Developer': df['Developer'],
        'Score': df['IGDB_Score'],
        'Category': df['Category_Primary'].fillna('Unclassified'), # Only the primary category
        'Release': df['Original_Release']
    })

    # 5. Save the final unnumbered Technical Performance spreadsheet
    df_bi_tech.to_excel(output_path, index=False)

    print("\n" + "═"*60)
    print(f"🚀 TECHNICAL ANALYSIS TEMPLATE GENERATED SUCCESSFULLY!")
    print(f"📊 Target shape: {df_bi_tech.shape[0]} rows x {df_bi_tech.shape[1]} columns.")
    print(f"💾 File saved for direct trend/historical analysis at: {output_path}")
    print("═"*60 + "\n")

    # Preview to verify the schema layout
    print("👀 PREVIEW OF THE TECHNICAL BI DATASET:")
    return df_bi_tech.head(5)

# Trigger the optimization
df_tech_preview = pipeline_bi_technical_analysis(INPUT_PATH, OUTPUT_PATH)